# Portfolio overview

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

train = pd.read_csv("../data/processed/train.csv")
validation = pd.read_csv("../data/processed/validation.csv")
test = pd.read_csv("../data/processed/test.csv")

In [ ]:
print(f"Train observations: {len(train):,}")
print(f"Validation observations: {len(validation):,}")
print(f"Test observations: {len(test):,}")

In [ ]:
train["SeriousDlqin2yrs"].mean()

In [ ]:
(
    train["SeriousDlqin2yrs"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Portfolio Overview

Total records: 90 000 in train sample

Bad Rate: 6.68%

Good Rate: 93.32%

# Variable Analysis

In [ ]:
summary = train.describe(
    percentiles=[0.01,0.05,0.25,0.5,0.75,0.95,0.99]
).T

summary

In [ ]:
# Missing Analysis

missing = pd.DataFrame({
    "missing_count": train.isna().sum(),
    "missing_pct": train.isna().mean() * 100
})

missing.sort_values(
    by="missing_pct",
    ascending=False
)

MonthlyIncome has approximately 19% missing values.

NumberOfDependents has approximately 2.57% missing values.

In [ ]:
# Distribution Analysis

def plot_distribution(df, variable):

    plt.figure(figsize=(8,4))

    df[variable].hist(
        bins=50
    )

    plt.title(variable)

    plt.show()

In [ ]:
plot_distribution(
    train,
    "age"
)

In [ ]:
plot_distribution(
    train,
    "MonthlyIncome"
)

In [ ]:
p99 = train["MonthlyIncome"].quantile(0.99)

train.loc[
    train["MonthlyIncome"] <= p99,
    "MonthlyIncome"
].hist(
    bins=50
)

plt.title("MonthlyIncome (<= P99)")
plt.show()

In [ ]:
plt.hist(
    np.log1p(train["MonthlyIncome"]),
    bins=50
)

plt.title("Log(MonthlyIncome)")
plt.show()

In [ ]:
plot_distribution(
    train,
    "DebtRatio"
)

In [ ]:
train["DebtRatio"].describe(
    percentiles=[0.5,0.75,0.9,0.95,0.99,0.999]
)

In [ ]:
p99 = train["DebtRatio"].quantile(0.99)

train.loc[
    train["DebtRatio"] <= p99,
    "DebtRatio"
].hist(
    bins=50
)

plt.title("DebtRatio (<= P99)")
plt.show()

In [ ]:
# log scale for monthly income
import numpy as np

plt.hist(
    np.log1p(train["DebtRatio"]),
    bins=5
)

plt.title("Log(DebtRatio)")
plt.show()

# Good vs Bad Comparison

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,5))

train.loc[
    train["SeriousDlqin2yrs"] == 0,
    "age"
].hist(
    bins=10,
    alpha=0.5,
    label="Good"
)

train.loc[
    train["SeriousDlqin2yrs"] == 1,
    "age"
].hist(
    bins=10,
    alpha=0.5,
    label="Bad"
)

plt.legend()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,5))

train.loc[
    train["SeriousDlqin2yrs"] == 0,
    "MonthlyIncome"
].hist(
    bins=10,
    alpha=0.5,
    label="Good"
)

train.loc[
    train["SeriousDlqin2yrs"] == 1,
    "MonthlyIncome"
].hist(
    bins=10,
    alpha=0.5,
    label="Bad"
)

plt.legend()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,5))

train.loc[
    train["SeriousDlqin2yrs"] == 0,
    "DebtRatio"
].hist(
    bins=10,
    alpha=0.5,
    label="Good"
)

train.loc[
    train["SeriousDlqin2yrs"] == 1,
    "DebtRatio"
].hist(
    bins=10,
    alpha=0.5,
    label="Bad"
)

plt.legend()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,5))

train.loc[
    train["SeriousDlqin2yrs"] == 0,
    "RevolvingUtilizationOfUnsecuredLines"
].hist(
    bins=10,
    alpha=0.5,
    label="Good"
)

train.loc[
    train["SeriousDlqin2yrs"] == 1,
    "RevolvingUtilizationOfUnsecuredLines"
].hist(
    bins=10,
    alpha=0.5,
    label="Bad"
)

plt.legend()
plt.show()

# Correlation Analysis

In [ ]:
# Pearson correlation

numeric_cols = train.select_dtypes(
    include=np.number
).columns

corr = train[
    numeric_cols
].corr()

corr

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,8))

plt.imshow(
    corr,
    cmap="coolwarm"
)

plt.colorbar()

plt.xticks(
    range(len(corr.columns)),
    corr.columns,
    rotation=90
)

plt.yticks(
    range(len(corr.columns)),
    corr.columns
)

plt.show()

In [ ]:
import numpy as np

corr_matrix = train.corr(numeric_only=True)

corr_pairs = (
    corr_matrix
    .where(
        np.triu(
            np.ones(corr_matrix.shape),
            k=1
        ).astype(bool)
    )
    .stack()
    .reset_index()
)

corr_pairs.columns = [
    "variable_1",
    "variable_2",
    "correlation"
]

high_corr = corr_pairs[
    abs(corr_pairs["correlation"]) > 0.7
]

high_corr.sort_values(
    by="correlation",
    ascending=False
)

In [ ]:
import numpy as np

corr_matrix = train.corr(numeric_only=True)

corr_pairs = (
    corr_matrix
    .where(
        np.triu(
            np.ones(corr_matrix.shape),
            k=1
        ).astype(bool)
    )
    .stack()
    .reset_index()
)

corr_pairs.columns = [
    "variable_1",
    "variable_2",
    "correlation"
]

high_corr = corr_pairs[
    abs(corr_pairs["correlation"]) > 0.7
]

high_corr.sort_values(
    by="correlation",
    ascending=False
)

EDA Findings

Strong multicollinearity observed between:

- NumberOfTime30-59DaysPastDueNotWorse
- NumberOfTimes90DaysLate
- NumberOfTime60-89DaysPastDueNotWorse

These variables will be reviewed during feature selection.

# stability checks for age variable

In [ ]:
train["age"].describe()

In [ ]:
validation["age"].describe()

In [ ]:
print(train.shape)
print(validation.shape)
print(test.shape)